# SVM Model - Online Gaming Behavior Prediction

## Problem Statement
Predict player engagement level (`Low`, `Medium`, `High`) using gameplay and player profile features from a real-world dataset.

## Member Contribution (SVM Part)
- Algorithm: Support Vector Machine (LinearSVC)
- Tasks: preprocessing integration, model training, hyperparameter tuning, evaluation, and analysis

## Dataset Source
Kaggle: Online Gaming Behavior Dataset

Use the exact dataset link in `submission.txt` and cite it in your report.

In [ ]:
# Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

sys.path.append(os.path.abspath(".."))
from src.preprocessing import preprocess_data

In [ ]:
# Load and preprocess data using shared project pipeline
X_train, X_test, y_train, y_test = preprocess_data(
    "../data/online_gaming_behavior_dataset.csv"
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

In [ ]:
# Baseline SVM (LinearSVC)
baseline_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42,
    max_iter=5000
)
baseline_svm.fit(X_train, y_train)

y_pred_baseline = baseline_svm.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred_baseline), 4))
print("Baseline Weighted F1:", round(f1_score(y_test, y_pred_baseline, average="weighted"), 4))

In [ ]:
# Hyperparameter tuning
param_grid = {
    "C": [0.1, 1.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
    "max_iter": [5000, 10000]
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_svm = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best CV Weighted F1:", round(grid.best_score_, 4))

In [ ]:
# Evaluate tuned model
y_pred = best_svm.predict(X_test)
decision_scores = best_svm.decision_function(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted")
rec = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print("========== FINAL SVM EVALUATION ==========")
print(f"Accuracy         : {acc:.4f}")
print(f"Precision        : {prec:.4f}")
print(f"Recall           : {rec:.4f}")
print(f"F1 Score         : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = sorted(y_test.unique())

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ROC AUC (One-vs-Rest) using decision scores
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

roc_auc = roc_auc_score(y_test_bin, decision_scores, average="macro", multi_class="ovr")
print(f"ROC AUC (OvR, macro): {roc_auc:.4f}")

In [ ]:
# ROC curves by class
plt.figure(figsize=(8, 6))

for i, class_label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], decision_scores[:, i])
    class_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_label} (AUC = {class_auc:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.title("SVM ROC Curve (One-vs-Rest)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Interpretability: top features by absolute linear coefficient magnitude
coef_abs_mean = np.mean(np.abs(best_svm.coef_), axis=0)
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": coef_abs_mean
}).sort_values("Importance", ascending=False)

top_k = 10
print(importance_df.head(top_k))

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df.head(top_k), x="Importance", y="Feature", palette="viridis")
plt.title("Top 10 Influential Features (Linear SVM)")
plt.tight_layout()
plt.show()

In [ ]:
# Sample prediction table for presentation/demo
sample_n = 12
sample_idx = X_test.sample(sample_n, random_state=42).index
sample_df = X_test.loc[sample_idx].copy()
sample_actual = y_test.loc[sample_idx]
sample_pred = best_svm.predict(sample_df)

result_df = pd.DataFrame({
    "Actual": sample_actual.values,
    "Predicted": sample_pred
})
result_df["Correct"] = result_df["Actual"] == result_df["Predicted"]

print(result_df)
print("\nSample Accuracy:", round(result_df["Correct"].mean() * 100, 2), "%")

## Discussion, Limitations, and Future Work
- The SVM model performs well on multiclass engagement prediction with balanced precision/recall.
- `LinearSVC` was chosen for scalability because the dataset is large.
- Possible improvements:
  - Kernel approximation (`Nystroem`) + linear classifier for non-linear decision boundaries
  - Feature selection and dimensionality reduction experiments
  - Cost-sensitive tuning based on business priorities (e.g., reducing false negatives for `High` engagement)
  - Probability calibration if calibrated class probabilities are required